# Runbook 3.5.b: Network monitoring and traffic analysis  (MODEL SOLUTION, instructor copy, do not distribute)

**Skill:** Detect Attacks · **Maps to:** 3.5.B · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** Implement a simplified network traffic-anomaly detector. You are given a per-host **baseline**
(the normal profile for that host) and a window of **observed flow records**. Flag every host that
either exceeds its baseline byte-volume threshold or uses a destination port not in its baseline's
allowed-port set.

A **baseline** is a dict mapping host IP (str) to a dict with:
- `"max_bytes"`: integer, the maximum total outbound bytes this host normally sends in the window
- `"allowed_ports"`: list of integers, destination ports this host normally uses

An **observed flow record** is a dict with:
- `"host"`: source host IP string
- `"dst_port"`: destination port integer
- `"bytes"`: integer, bytes sent in this flow

A host may appear in multiple flow records within the same window; volumes accumulate.

**Lesson objective.** Explain how baseline traffic profiles support anomaly detection, read basic flow-record and packet-capture data, and identify network-monitoring indicators of compromise such as beaconing, unusual port usage, and abnormal data-volume spikes.

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.

### Worked examples
```python
baseline = {
    "10.0.1.5": {"max_bytes": 500_000_000, "allowed_ports": [445, 139]},
    "10.0.2.9": {"max_bytes": 5_000_000, "allowed_ports": [80, 443]},
}
flows = [
    {"host": "10.0.1.5", "dst_port": 445, "bytes": 100_000_000},
    {"host": "10.0.1.5", "dst_port": 445, "bytes": 50_000_000},
    {"host": "10.0.2.9", "dst_port": 4444, "bytes": 1_000},   # unusual port -> flagged
]
solve(baseline, flows) == ["10.0.2.9"]
```

```python
baseline = {"10.0.3.1": {"max_bytes": 1_000_000, "allowed_ports": [443]}}
flows = [
    {"host": "10.0.3.1", "dst_port": 443, "bytes": 900_000},
    {"host": "10.0.3.1", "dst_port": 443, "bytes": 900_000},   # total 1.8M > 1M -> flagged
]
solve(baseline, flows) == ["10.0.3.1"]
```

- [ ] Work the first example by hand and get the same answer

### Rules to respect
- [ ] Deterministic, offline, standard library only.
- [ ] Must pass all cases in the tests (visible + hidden).
- [ ] Result is sorted, contains no duplicate host IPs.
- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry point
```python
def solve(baseline: dict, flows: list[dict]) -> list[str]
```

#### Input

##### `baseline`, dict mapping host IP (str) -> profile dict
Each profile dict has:
- `"max_bytes"` (int): maximum normal total outbound bytes for this host in the window
- `"allowed_ports"` (list[int]): destination ports normally used by this host

##### `flows`, list of observed flow record dicts (any order, possibly multiple per host)
Each flow dict has:
- `"host"` (str): source host IP
- `"dst_port"` (int): destination port
- `"bytes"` (int): bytes sent in this flow

#### Output (`result`)
Sorted `list[str]` of host IPs flagged as anomalous, no duplicates.

#### Flagging rules
A host is flagged if ANY of the following hold:
1. Host is not a key in `baseline` (unknown host, no profile to compare against).
2. The sum of `"bytes"` across all of that host's flow records exceeds `baseline[host]["max_bytes"]`.
3. Any single flow record for that host has a `"dst_port"` not in `baseline[host]["allowed_ports"]`.

#### Guarantees / rules
- Pure function; deterministic; no I/O; no network access.
- `baseline` and `flows` are not mutated by `solve`.
- Standard library only.

#### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).

</details>

## 3. Build it

This is the model solution.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
from typing import Dict, List


def solve(baseline: Dict, flows: List[Dict]) -> List[str]:
    totals: Dict[str, int] = {}
    port_violation: Dict[str, bool] = {}
    unknown_hosts = set()

    for flow in flows:
        host = flow["host"]
        if host not in baseline:
            unknown_hosts.add(host)
            continue
        totals[host] = totals.get(host, 0) + flow["bytes"]
        allowed_ports = baseline[host]["allowed_ports"]
        if flow["dst_port"] not in allowed_ports:
            port_violation[host] = True

    flagged = set(unknown_hosts)
    for host, total in totals.items():
        if total > baseline[host]["max_bytes"]:
            flagged.add(host)
    for host in port_violation:
        flagged.add(host)

    return sorted(flagged)


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

def test_port_anomaly_flags_host():
    baseline = {
        "10.0.1.5": {"max_bytes": 500_000_000, "allowed_ports": [445, 139]},
        "10.0.2.9": {"max_bytes": 5_000_000, "allowed_ports": [80, 443]},
    }
    flows = [
        {"host": "10.0.1.5", "dst_port": 445, "bytes": 100_000_000},
        {"host": "10.0.1.5", "dst_port": 445, "bytes": 50_000_000},
        {"host": "10.0.2.9", "dst_port": 4444, "bytes": 1_000},
    ]
    assert impl.solve(baseline, flows) == ["10.0.2.9"]


def test_volume_anomaly_flags_host():
    baseline = {"10.0.3.1": {"max_bytes": 1_000_000, "allowed_ports": [443]}}
    flows = [
        {"host": "10.0.3.1", "dst_port": 443, "bytes": 900_000},
        {"host": "10.0.3.1", "dst_port": 443, "bytes": 900_000},
    ]
    assert impl.solve(baseline, flows) == ["10.0.3.1"]


def test_normal_traffic_not_flagged():
    baseline = {"10.0.5.5": {"max_bytes": 1_000_000, "allowed_ports": [80, 443]}}
    flows = [
        {"host": "10.0.5.5", "dst_port": 443, "bytes": 400_000},
        {"host": "10.0.5.5", "dst_port": 80, "bytes": 200_000},
    ]
    assert impl.solve(baseline, flows) == []


def test_unknown_host_flagged():
    baseline = {"10.0.1.1": {"max_bytes": 1_000_000, "allowed_ports": [80]}}
    flows = [
        {"host": "10.0.1.1", "dst_port": 80, "bytes": 100},
        {"host": "192.168.9.9", "dst_port": 80, "bytes": 100},
    ]
    assert impl.solve(baseline, flows) == ["192.168.9.9"]


def test_exact_boundary_not_flagged():
    baseline = {"10.0.6.1": {"max_bytes": 1_000, "allowed_ports": [80]}}
    flows = [{"host": "10.0.6.1", "dst_port": 80, "bytes": 1_000}]
    assert impl.solve(baseline, flows) == []


def test_multiple_hosts_mixed_results():
    baseline = {
        "a": {"max_bytes": 100, "allowed_ports": [22]},
        "b": {"max_bytes": 100, "allowed_ports": [22]},
        "c": {"max_bytes": 100, "allowed_ports": [22]},
    }
    flows = [
        {"host": "a", "dst_port": 22, "bytes": 50},
        {"host": "b", "dst_port": 22, "bytes": 150},
        {"host": "c", "dst_port": 9999, "bytes": 10},
    ]
    assert impl.solve(baseline, flows) == ["b", "c"]


def test_empty_flows():
    baseline = {"a": {"max_bytes": 100, "allowed_ports": [22]}}
    assert impl.solve(baseline, []) == []


def test_result_sorted_and_deduplicated():
    baseline = {"z": {"max_bytes": 10, "allowed_ports": [80]}, "a": {"max_bytes": 10, "allowed_ports": [80]}}
    flows = [
        {"host": "z", "dst_port": 999, "bytes": 1},
        {"host": "z", "dst_port": 999, "bytes": 1},
        {"host": "a", "dst_port": 999, "bytes": 1},
    ]
    assert impl.solve(baseline, flows) == ["a", "z"]


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Call `solve` on an input you make up, and predict the answer first.

- [ ] Your prediction matched

## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it